In [1]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader(r"..\data\Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [4]:
# uv add faiss-cpu
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [5]:
vectorstore.save_local("./faiss_index")

In [6]:
vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [7]:
query = "결혼하면 얼마를 받을 수 있을까?"

In [8]:
results = vectorstore.similarity_search(query, k=3)

In [9]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [10]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [11]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

chain = prompt | model | parser

In [12]:
res = chain.invoke({"context": results, "question": query})

In [13]:
res

'결혼 시 본인에게는 100만원의 경조금과 화환이 지원됩니다.'

---

In [14]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
model.invoke(query)

AIMessage(content="결혼하면 얼마를 받을 수 있는지 궁금하시군요. 이는 **결혼 축하금, 혼수, 신혼여행 비용 등 다양한 형태로 지원받을 수 있는 부분**을 의미하는 것으로 이해됩니다.\n\n결혼 관련 지원금은 개인의 상황, 지역, 직장, 결혼식 규모 등에 따라 천차만별이며, 법적으로 정해진 금액이 있는 것은 아닙니다. 하지만 일반적으로 다음과 같은 경우에 지원이나 혜택을 받을 수 있습니다.\n\n**1. 정부 및 지자체 지원:**\n\n*   **주거 지원:** 신혼부부 특별 공급 주택, 주택 구입/전세 자금 대출 우대 금리 등 정부나 지자체에서 제공하는 주거 관련 혜택이 있습니다. 이는 직접적인 현금 지급은 아니지만, 결혼 후 주거 마련에 드는 비용을 크게 절감할 수 있습니다.\n*   **출산/육아 지원:** 결혼 후 자녀 계획이 있다면, 출산 장려금, 아동 수당 등 정부 및 지자체의 육아 관련 지원금을 받을 수 있습니다.\n\n**2. 직장 지원:**\n\n*   **결혼 휴가 및 축하금:** 일부 기업에서는 직원의 결혼을 축하하기 위해 일정 기간의 유급 휴가를 제공하거나, 소정의 결혼 축하금을 지급하기도 합니다. 이는 회사마다 정책이 다르므로 해당 직장에 문의해보는 것이 좋습니다.\n\n**3. 가족 및 친척의 축의금:**\n\n*   결혼식에 참석하는 하객들이 축의금을 전달하는 것이 일반적입니다. 축의금의 액수는 관계의 친밀도, 경제적 상황 등에 따라 다르지만, 결혼 비용 마련에 큰 도움이 될 수 있습니다.\n\n**4. 혼수 및 예물:**\n\n*   결혼 준비 과정에서 신랑, 신부 혹은 양가 부모님께서 혼수(생활에 필요한 물품)나 예물(반지, 목걸이 등)을 준비해 주시는 경우가 많습니다. 이는 직접적인 현금 지급은 아니지만, 결혼 생활에 필요한 자원을 확보하는 것입니다.\n\n**5. 신혼여행 비용:**\n\n*   일부 부모님이나 가족분들이 신혼여행 비용을 지원해주시는 경우도 있습니다.\n\n**결론적으로, '결혼하면 얼마를 받을